# Geomechanical Injection Scenario Toolkit (GIST)

#Disclaimer
GIST aims to give the _gist_ of a wide range of potential scenarios and aid collective decision making when responding to seismicity.

The results of GIST are entirely dependent upon the inputs provided, which may be incomplete or inaccurate.

There are other potentially plausible inducement scenarios that are not considered, including fluid migration into the basement, 
out-of-zone poroelastic stressing, or hydraulic fracturing.

None of the individual models produced by GIST accurately represent what happens in the subsurface and cannot be credibly used 
to accurately assign liability or responsibility for seismicity.

"All models are wrong, but some are useful" - George Box, 1976

## Prerequisites

Assumes InjectionSQLScheduled completed successfully and injection data are sampled uniformly in time

In [0]:
%run "./GIST_RunTemplate_ErrorReporting_Step1"

# Geomechanical Injection Scenario Toolkit (GIST)

#Disclaimer
GIST aims to give the _gist_ of a wide range of potential scenarios and aid collective decision making when responding to seismicity.

The results of GIST are entirely dependent upon the inputs provided, which may be incomplete or inaccurate.

There are other potentially plausible inducement scenarios that are not considered, including fluid migration into the basement, 
out-of-zone poroelastic stressing, or hydraulic fracturing.

None of the individual models produced by GIST accurately represent what happens in the subsurface and cannot be credibly used 
to accurately assign liability or responsibility for seismicity.

"All models are wrong, but some are useful" - George Box, 1976

## Prerequisites

Assumes InjectionSQLScheduled completed successfully and injection data are sampled uniformly in time

##Install Dependencies
- geopandas
- gistMC.py
- eqSQL.py
- gistPlots.py
- numpy
- scipy
- pandas


##Paths

##Libraries

- numpy
- scipy
- pandas
- matplotlib
- geopandas
- pyspark


#1. Select Event

##1.1. Parameters: TexNet event ID + forecast duration

##1.2. Results: Create directories, fetch EQ

#2. Initial Run

##2.1. Parameters
###2.1.1: Subsurface Modeling

In [0]:
# Binary deep/shallow parameter
deepOrShallow='Deep'
# Depth from surface (ft)
# Distinguishes deep/shallow where we don't have a horizon 
depthCutoff=8000.

In [0]:
nRealizations=250

In [0]:
# Porosity in percent
PorosityPercentMin=3.
PorosityPercentMax=12.

# Permeability in millidarcies
PermMDMin=20.
PermMDMax=400.

# Thickness in feet
ThicknessFTMin=200.
ThicknessFTMax=500.

# Vertical compressibility minimum/maximum (1/Pa)
VerticalCompressibilityMin=0.00000000100
VerticalCompressibilityMax=0.00000000110

In [0]:
# Water density minimum/maximum (kg/m3)
WaterDensityMin=1000.
WaterDensityMax=1050.
# Water viscosity minimum/maximum (Pa.s)
WaterViscosityMin=0.000799
WaterViscosityMax=0.000801
# Fluid Compressibility minimum/maximum (1/Pa)
FluidCompressibilityMin=0.000000000359
FluidCompressibilityMax=0.000000000361

In [0]:
# Oversampling of time axis (poroelastic-only)
nTimeBins=21
# Poroelastic parameters - for in-zone poroelasticity in v2
ShearModulusMin=4e9
ShearModulusMax=6e9
PoissonsRatioDrainedMin=0.295
PoissonsRatioDrainedMax=0.305
PoissonsRatioUndrainedMin=0.305
PoissonsRatioUndrainedMax=0.315
BiotsCoefficientMin=0.26
BiotsCoefficientMax=0.36
# Fault parameters (poroelastic-only)
FaultFrictionCoeffMin=0.55
FaultFrictionCoeffMax=0.65
RockFrictionCoeffMin=0.55
RockFrictionCoeffMax=0.65

###2.1.2: Output Filtering

In [0]:
# Verbosity - 0=silent, 1=some, 2=lots
verb=0
# Minimum Pressure change to care about in PSI
dPCutoff=1.
# Maximum Number of wells to plot
nWells=30
# What is the first year we plot?
minYear=1980

##2.2. Results:
### 2.2.1: Set Interval-Specific Paths


In [0]:
# Set an output directory for this earthquake and this interval
runIntervalPath=runPath+deepOrShallow+'/'
initialRunIntervalPath=runIntervalPath+'initialRun/'
updatedRunIntervalPath=runIntervalPath+'udpatedRun/'
forecastRunIntervalPath=runIntervalPath+'forecastRun/'
disposalPath=runIntervalPath+'updatedDisposal/'
# Make directory if it doesn't exist
os.makedirs(runIntervalPath, exist_ok=True)
os.makedirs(initialRunIntervalPath, exist_ok=True)
os.makedirs(updatedRunIntervalPath, exist_ok=True)
os.makedirs(forecastRunIntervalPath, exist_ok=True)
os.makedirs(disposalPath, exist_ok=True)
os.makedirs(initialRunIntervalPath+'perWell', exist_ok=True)
os.makedirs(updatedRunIntervalPath+'perWell', exist_ok=True)
os.makedirs(forecastRunIntervalPath+'perWell', exist_ok=True)
################################################
# Output prefix for realizations of parameters #
################################################
RealizationPrefix=runIntervalPath+'MC'

In [0]:
# Point to appropriate well and injection files from initial database
if deepOrShallow=='Deep':
  WellFile=injPath+'/deep.csv'
  InjFile=injPath+'/deepReg.csv'
elif deepOrShallow=='Shallow':
  WellFile=injPath+'/shallow.csv'
  InjFile=injPath+'/shallowReg.csv'

###2.2.2: Initialize and Select Wells

In [0]:
gist=gi.gistMC(nReal=nRealizations,
                ntBin=nTimeBins)
gist.initPP(rho0_min=WaterDensityMin,
             rho0_max=WaterDensityMax,
             phi_min=PorosityPercentMin,
             phi_max=PorosityPercentMax,
             kMD_min=PermMDMin,
             kMD_max=PermMDMax,
             h_min=ThicknessFTMin,
             h_max=ThicknessFTMax,
             alphav_min=VerticalCompressibilityMin,
             alphav_max=VerticalCompressibilityMax,
             beta_min=FluidCompressibilityMin,
             beta_max=FluidCompressibilityMax)

In [0]:
gist.addWells(wellFile=WellFile,injFile=InjFile,verbose=verb)
if verb>0:
  gist.wellDF.info()
  EQDF.info()

In [0]:
print(eq.info)
print(EQDF.info())

In [0]:
selectedWellsDF,ignoredWellsDF,injDF=gist.findWellsVec(EQDF,PE=False,responseYears=forecastYears,verbose=verb)
# Isolate wells included only for forecast
currentWellsDF=selectedWellsDF[selectedWellsDF['EncompassingDay']<0.].reset_index(drop=True)
forecastWellsDF=selectedWellsDF[selectedWellsDF['EncompassingDay']>=0.].reset_index(drop=True)
print(selectedWellsDF.shape,currentWellsDF.shape,forecastWellsDF.shape)

In [0]:
print(selectedWellsDF.columns)
print(currentWellsDF.columns)

###2.2.3: Calculate Pressure Ranges of Selected Wells

In [0]:
scenarioDF=gist.runPressureScenariosVec(EQDF,currentWellsDF,injDF,verbose=1)

In [0]:
filteredDF,orderedWellIDList=gi.summarizePPResults(scenarioDF,currentWellsDF,threshold=dPCutoff,nOrder=nWells,verbose=2)

###2.2.4: Generate selection and disaggregation plots

In [0]:
disaggregationDF=gi.prepDisaggregationPlot(filteredDF,orderedWellIDList,jitter=0.1,verbose=0)

In [0]:
diffRange=(min(gist.diffPPVec),max(gist.diffPPVec))
rtDF,mergedWellsDF = gi.prepRTPlot(selectedWellsDF,ignoredWellsDF,minYear,diffRange,eq,clipYear=False,verbose=1)

###2.2.5: Generate Time Series Pressures for significant wells

In [0]:
winWellsDF,winInjDF=gi.getWinWells(filteredDF,currentWellsDF,injDF)
display(winWellsDF)

In [0]:

scenarioTSRDF,dPTimeSeriesR,wellIDsR,dayVecR = gist.runPressureScenariosTimeSeriesConv(EQDF,winWellsDF,winInjDF,verbose=verb)

In [0]:
totalPPQuantilesDF=gi.prepTotalPressureTimeSeriesQuantilesPlot(dPTimeSeriesR,dayVecR,nQuantiles=21,epoch=pd.to_datetime('1970-01-01'),verbose=1)
totalPPSpaghettiDF=gi.prepTotalPressureTimeSeriesSpaghettiPlot(dPTimeSeriesR,dayVecR,gist.diffPPVec,epoch=pd.to_datetime('1970-01-01'),verbose=1)
print(totalPPQuantilesDF)

In [0]:
allPPQuantilesDF,allPPSpaghettiDF=gi.getPerWellPressureTimeSeriesSpaghettiAndQuantiles(dPTimeSeriesR,dayVecR,gist.diffPPVec,wellIDsR,nQuantiles=11,epoch=pd.to_datetime('01-01-1970'))

In [0]:
wellPressureDict=gi.prepPressureAndDisposalTimeSeriesPlots(allPPQuantilesDF,allPPSpaghettiDF,winWellsDF,winInjDF,orderedWellIDList[:-1],verbose=0)

###2.2.6: Sensitivity Analysis, Generate Tornado Plots

In [0]:
# Calculate sensitivities to different parameters
sensitivityDF,sensitivitySumDF = gist.getPressureSensitivity(winInjDF,winWellsDF,EQDF,verbose=1)
# I'm not getting negative pressures anymore
print(sensitivitySumDF)

In [0]:
# This is the date to calcaulte THdPdT0
futureEQ=eq.copy()
futureEQ.loc['Origin Date']=pd.to_datetime('2026-9-1')
print(eq,futureEQ)
rateDF = gist.getTHdPdT0(winWellsDF,winInjDF,eq,futureEQ,2)

###2.2.7: Output CSV files

In [0]:
gist.writeRealizations(initialRunIntervalPath+'PorePressureRealizations.csv')

In [0]:
scenarioDF.to_csv(initialRunIntervalPath+'scenarios.csv')

In [0]:
filteredDF.to_csv(initialRunIntervalPath+'filteredScenarios.csv')
pd.Series(data=orderedWellIDList).to_csv(initialRunIntervalPath+'wellIDOrder.csv')

In [0]:
mergedWellsDF.to_csv(initialRunIntervalPath+'RTwells.csv')
rtDF.to_csv(initialRunIntervalPath+'RTDF.csv')

In [0]:
disaggregationDF.to_csv(initialRunIntervalPath+'disaggregation.csv')

In [0]:
totalPPQuantilesDF.to_csv(initialRunIntervalPath+'totalPPQuantiles.csv')
totalPPSpaghettiDF.to_csv(initialRunIntervalPath+'totalPPSpaghetti.csv')

In [0]:
i=0
for wellDictKey, wellDictValue in wellPressureDict.items():
  wellID=wellDictValue['WellInfo']['ID'].to_list()[0]
  wellFilePrefix='/perWell/well_'+str(i)+'_'
  wellDictValue['PPQuantiles'].to_csv(initialRunIntervalPath+wellFilePrefix+'PPQuantiles.csv')
  wellDictValue['Disposal'].to_csv(initialRunIntervalPath+wellFilePrefix+'Disposal.csv')
  wellDictValue['Spaghetti'].to_csv(initialRunIntervalPath+wellFilePrefix+'Spaghetti.csv')
  wellDictValue['WellInfo'].to_csv(initialRunIntervalPath+wellFilePrefix+'WellInfo.csv')
  print('well',i,', ID:',wellID,' completed')
  i=i+1

In [0]:
scenarioTSRDF.to_csv(initialRunIntervalPath+'materialScenariosR.csv')
np.savez_compressed(initialRunIntervalPath+'timeSeriesR.npz', deltaPP=dPTimeSeriesR,dayVec=dayVecR,wellIDs=wellIDsR)

In [0]:
sensitivityDF.to_csv(initialRunIntervalPath+'sensitivity.csv')
sensitivitySumDF.to_csv(initialRunIntervalPath+'sensitivitySum.csv')

In [0]:
rateDF.to_csv(initialRunIntervalPath+'rates.csv')

#3. Correct Data

##3.1 Export Disposal Data

In [0]:
selectedWellsDF.to_csv(disposalPath+'selectedWells.csv')
ignoredWellsDF.to_csv(disposalPath+'ignoredWells.csv')
allWellsDF=pd.concat([selectedWellsDF,ignoredWellsDF])
allWellsDF.to_csv(disposalPath+'allInZoneWells.csv')
injDF.to_csv(disposalPath+'inj.csv')
injDF.to_csv(disposalPath+'inj.zip',compression='zip')

In [0]:
# End Year of time series forecast - I don't think that I need this now.
#EndYear=2030
# Rerun select wells with an updated earthquake time set to the last day of EndYear
#forecastEQ=eq.copy()
#forecastEQ['Origin Date']=pd.to_datetime('12-31-'+str(EndYear))
#selectedForecastWellsDF,ignoredForecastWellsDF,injDF=gist.findWells(forecastEQ,PE=False,verbose=verb)
##selectedWellsDF.to_csv(disposalPath+'forecastSelectedWells.csv')
#ignoredWellsDF.to_csv(disposalPath+'forecastIgnoredWells.csv')
#allWellsDF=pd.concat([selectedWellsDF,ignoredWellsDF])
#allWellsDF.to_csv(disposalPath+'forecastAllInZoneWells.csv')
#injDF.to_csv(disposalPath+'forecastPastInj.csv')

##3.2 Import Updated Disposal Data

In [0]:
# I need to merge the updated disposal information with the prior stuff
# Load updated injection file
# Well comparisons
# Get list of well IDs in the updated injection file
# Get list of well IDs in updated well file
# Check to see overlap in well IDs
#    updated vs. prior selected wells
#    updated vs. prior ignored wells
# FilteredIgnored =  prior ignored wells - (wells in updated file and prior ignored wells)
# Merge FilteredIgnored with with updated wells
# Load injection
#    Check for time sampling of injection - all wells must be the same
#    Get overall time vector - convert to days
#    Loop over wells in well file
#      Get all disposal data for well ID
#      Convert date values to days
#      
#mergedInjFile=